# Omni-VLA: CFM on dataset v2
Attach the NEW source ZIP and drawer_dagger_v2.h5. Enable GPU + Internet. Train CFM only; BC v2 stays the frozen baseline. Independent-pair CFM, chunk 16, primary execute horizon 1; CPU evaluation uses 4/6/8 Euler steps on seeds 5000-5049 and 7000-7049.


In [ ]:
from pathlib import Path
import zipfile, sys, subprocess, shutil
import torch

POLICIES = ("cfm",)  # CFM only; frozen BC v2 is evaluated on Dell.
INPUT = Path("/kaggle/input")
REPO = Path("/kaggle/working/omni_vla")
print(torch.__version__, "CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings > Accelerator"
print("GPU:", torch.cuda.get_device_name(0))

# Kaggle may expose the ZIP itself or unpack its contents during dataset upload.
archives = list(INPUT.rglob("kaggle_source.zip"))
if archives:
    assert len(archives) == 1, "Attach exactly one source archive"
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall("/kaggle/working")
else:
    candidates = [p.parent for p in INPUT.rglob("pyproject.toml")
                  if (p.parent / "scripts/train_kaggle.py").is_file()]
    assert len(candidates) == 1, "Attach the Omni-VLA source dataset (ZIP or unpacked files)"
    shutil.copytree(candidates[0], REPO, dirs_exist_ok=True)
assert (REPO / "scripts/train_kaggle.py").is_file(), "Source layout is incomplete"
assert (REPO / "configs/train_cfm_dagger_v2.yaml").is_file(), "Upload the latest CFM source ZIP"
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)+"[train,deploy]"], check=True)


In [ ]:
datasets = list(INPUT.rglob("drawer_dagger_v2.h5"))
assert len(datasets) == 1, "Attach exactly one drawer_dagger_v2.h5 dataset"
DATA = datasets[0]
print("Dataset:", DATA)
subprocess.run([sys.executable, "-m", "scripts.validate_dataset", str(DATA)], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    output = Path(f"/kaggle/working/{policy}")
    assert not output.exists(), "Use a fresh session or a new output path to keep run logs separate"
    subprocess.run([sys.executable, "-m", "scripts.train_kaggle", "--dataset", str(DATA), "--policy", policy, "--config", str(REPO / "configs/train_cfm_dagger_v2.yaml"), "--output", str(output)], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    subprocess.run([sys.executable, "-m", "scripts.export_policy", "--checkpoint", f"/kaggle/working/{policy}/best.pt", "--output", f"/kaggle/working/{policy}/policy.onnx"], cwd=REPO, check=True)


In [ ]:
for policy in POLICIES:
    archive = shutil.make_archive(f"/kaggle/working/{policy}_dagger_v2_outputs", "zip",
                                  root_dir="/kaggle/working", base_dir=policy)
    print("Download:", archive)
    print("Checkpoint:", Path(f"/kaggle/working/{policy}/best.pt"))


Save Version with outputs, download cfm_dagger_v2_outputs.zip. On Dell: `.venv/bin/python -m scripts.evaluate_cfm_candidate --archive cfm_dagger_v2_outputs.zip`. This imports a new CFM checkpoint, checks ONNX parity, and runs 300 CPU trials (2 seed sets x 3 Euler settings x 50 episodes). GPU training and CFM task quality are unverified until executed.
